# 🪱 Helminths — RESUME from Drive checkpoint (continue-from-weights)

**Runtime → Change runtime type → T4 GPU.** Run cells top → bottom.

Uses continue-from-weights (NOT `resume=True`) on the helminth-egg dataset, and checkpoints to Drive every 5 epochs so a disconnect never loses progress.

> Injira kuri account ifite `nexus_helminths/.../last.pt` (cyangwa `nexus_parasitology/` ya kera) kuri Drive + GPU.

## 1. Install

In [ ]:
!pip -q install ultralytics roboflow
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| cuda', torch.cuda.is_available())

## 2. Resume (checkpoint + helminth data + continue)

In [ ]:
import os, glob, yaml
from getpass import getpass
from google.colab import drive; drive.mount('/content/drive')
from ultralytics import YOLO

# 1) latest checkpoint on Drive (nexus_helminths, or legacy nexus_parasitology)
roots = ['/content/drive/MyDrive/nexus_helminths', '/content/drive/MyDrive/nexus_parasitology']
cks = sorted([p for r in roots
              for p in glob.glob(r + '/**/last.pt', recursive=True) + glob.glob(r + '/**/best.pt', recursive=True)
              if 'yolov8' not in os.path.basename(p).lower()], key=os.path.getmtime)
assert cks, 'Nta last.pt kuri Drive/nexus_helminths (cyangwa nexus_parasitology).'
ckpt = cks[-1]; names = list(YOLO(ckpt).model.names.values())
print('checkpoint:', ckpt, '\nclasses:', names)
assert any(k in str(n).lower() for n in names for k in ('ascaris','hookworm','trichuris','taenia')), \
    'Iyi checkpoint SI iya helminth eggs.'

# 2) helminth-egg dataset (Roboflow) + align names -> checkpoint
os.environ['ROBOFLOW_API_KEY'] = getpass('Roboflow key: ').strip()
from roboflow import Roboflow
proj = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY']).workspace('parasites-onxa6').project('parasites-trgfz')
ds = None
for v in range(1, 12):
    try: ds = proj.version(v).download('yolov8', location='/content/data/para_rf'); break
    except Exception: continue
assert ds, 'Roboflow download yanze.'
DATA_YAML = ds.location + '/data.yaml'
d = yaml.safe_load(open(DATA_YAML)); dn = d['names']
dn = [dn[k] for k in sorted(dn)] if isinstance(dn, dict) else list(dn)
if len(dn) == len(names):
    d['names'] = names; d['nc'] = len(names); yaml.safe_dump(d, open(DATA_YAML, 'w'), sort_keys=False)
print('DATA_YAML =', DATA_YAML, '| classes:', d['names'])

# 3) CONTINUE-FROM-WEIGHTS (NTA resume=True) -> ibika Drive buri 5 epochs
YOLO(ckpt).train(data=DATA_YAML, epochs=40, imgsz=640, batch=16,
    project='/content/drive/MyDrive/nexus_helminths/runs', name='helminths_finish',
    pretrained=True, patience=20, plots=True, save_period=5,
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, degrees=180, fliplr=0.5, flipud=0.5,
    mosaic=1.0, translate=0.1, scale=0.5)
m = YOLO('/content/drive/MyDrive/nexus_helminths/runs/helminths_finish/weights/best.pt').val(data=DATA_YAML)
print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')

## 3. Export helminths.pt

In [ ]:
# 4) Export helminths.pt (Drive + download)
import glob, os, shutil
best = sorted(glob.glob('/content/drive/MyDrive/nexus_helminths/runs/**/weights/best.pt', recursive=True),
              key=os.path.getmtime)[-1]
print('best:', best)
shutil.copy(best, '/content/drive/MyDrive/nexus_helminths/helminths.pt')
from google.colab import files; files.download(best)